# 🩸 혈액세포 이미지 다운로드 (BloodMNIST)

Hugging Face의 **OctoMed/BloodMNIST** 데이터셋에서 **Neutrophil(호중구)** 과 **Lymphocyte(림프구)** 이미지를 받아
Teachable Machine에 바로 올릴 수 있는 zip 파일로 만들어 줍니다.

## 사용법 (3단계)
1. 위 메뉴에서 **런타임 → 모두 실행** 을 누르세요. (단축키 `Ctrl + F9`)
2. "이 노트북은 Google에서 작성하지 않았습니다" 경고가 나오면 **무시하고 계속** 을 누르세요.
3. 1~2분 기다리면 `bloodmnist.zip` 파일이 자동으로 다운로드됩니다.

> 압축을 풀면 `neutrophil` 폴더와 `lymphocyte` 폴더가 나옵니다.
> Teachable Machine에서 클래스 이름을 각각 정하고, 폴더 안 이미지를 모두 끌어다 놓으면 됩니다.

## ⚙️ 설정
몇 장씩 받을지 바꾸고 싶으면 아래 숫자만 고치세요.

In [ ]:
COUNT = 30  # 클래스당 이미지 수

## 1️⃣ 이미지 다운로드

In [ ]:
import os, shutil
from datasets import load_dataset, ClassLabel

# BloodMNIST 표준 라벨 번호
MEDMNIST_LABELS = {0: "basophil", 1: "eosinophil", 2: "erythroblast", 3: "immature granulocytes",
                   4: "lymphocyte", 5: "monocyte", 6: "neutrophil", 7: "platelet"}
TARGETS = ["neutrophil", "lymphocyte"]
OUT = "bloodmnist"

def find_column(features, kind):
    for name, feature in features.items():
        if kind == "image" and feature.__class__.__name__ == "Image":
            return name
        if kind == "label" and name.lower() in ("label", "labels", "class", "answer"):
            return name
    raise SystemExit(f"{kind} 컬럼을 찾지 못했습니다. 컬럼 목록: {list(features)}")

def label_to_name(value, feature):
    if isinstance(value, list):
        value = value[0]
    if isinstance(feature, ClassLabel):
        return feature.int2str(int(value)).lower()
    if isinstance(value, int):
        return MEDMNIST_LABELS.get(value, str(value))
    return str(value).lower()

shutil.rmtree(OUT, ignore_errors=True)
for t in TARGETS:
    os.makedirs(os.path.join(OUT, t))

ds = load_dataset("OctoMed/BloodMNIST", split="train", streaming=True)
image_col = find_column(ds.features, "image")
label_col = find_column(ds.features, "label")

saved = {t: 0 for t in TARGETS}
for row in ds:
    name = label_to_name(row[label_col], ds.features[label_col])
    target = next((t for t in TARGETS if t in name), None)
    if target is None or saved[target] >= COUNT:
        continue
    row[image_col].convert("RGB").save(os.path.join(OUT, target, f"{target}_{saved[target]:03d}.png"))
    saved[target] += 1
    if all(n >= COUNT for n in saved.values()):
        break

print("✅ 저장 완료:", saved)

## 2️⃣ 받은 이미지 미리보기

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for r, t in enumerate(TARGETS):
    files = sorted(os.listdir(os.path.join(OUT, t)))[:8]
    for c, f in enumerate(files):
        axes[r, c].imshow(Image.open(os.path.join(OUT, t, f)))
        axes[r, c].axis("off")
    axes[r, 0].set_title(t, loc="left", fontsize=14)
plt.tight_layout()
plt.show()

## 3️⃣ zip 파일로 내 컴퓨터에 저장
자동으로 다운로드되지 않으면 왼쪽 📁 파일 탭에서 `bloodmnist.zip` 을 오른쪽 클릭 → **다운로드** 하세요.

In [ ]:
shutil.make_archive("bloodmnist", "zip", OUT)
from google.colab import files
files.download("bloodmnist.zip")